In [5]:
import wandb
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import hydra
from hydra import initialize, compose
from omegaconf import OmegaConf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Dense, Input, GlobalAveragePooling2D, MaxPooling2D, Flatten, BatchNormalization, Dropout
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2
from tensorflow.keras import optimizers
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau
from loss import compute_class_weights, set_binary_crossentropy_weighted_loss
from utils import NumpyEncoder, plot_metrics

In [3]:
wandb.init(project="chexpert")

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: nityanandmathur. Use `wandb login --relogin` to force relogin


In [8]:
initialize(version_base=None, config_path="conf/")
cfg=compose(config_name="config.yaml")
print(OmegaConf.to_yaml(cfg))

params:
  batch_size: 32,
  learning_rate: 0.01,
  training_epoch: 30,
  num_gpus: 4



In [9]:
batch_size = cfg.params.batch_size
num_gpus = cfg.params.num_gpus
learning_rate = cfg.params.learning_rate
training_epochs= cfg.params.learning_rate

# config = {
#     "batch_size": 32,
#     "learning_rate": 0.01,
#     "training_epoch": 30,
#     "num_gpus": 4
# }

In [4]:
#Data Directory and CSV files containing info about data
data_dir = "/home/btech/nityanand.mathur/chexpert/"
train_df = pd.read_csv(filepath_or_buffer="/home/btech/nityanand.mathur/chexpert/labels/train_val_split_data/train_u-zeros.csv", dtype={
    "Path": str,
    "Atelectasis": np.float32,
    "Cardiomegaly": np.float32,
    "Consolidation": np.float32,
    "Edema": np.float32,
    "Pleural Effusion": np.float32,
    "Pleural Other": np.float32,
    "Pneumonia": np.float32,
    "Pneumothorax": np.float32,
    "Enlarged Cardiomediastinum": np.float32,
    "Lung Opacity": np.float32,
    "Lung Lesion": np.float32,
    "Fracture": np.float32,
    "Support Devices": np.float32,
    "No Finding": np.float32
})
val_df = pd.read_csv(filepath_or_buffer="/home/btech/nityanand.mathur/chexpert/labels/train_val_split_data/val_u-zeros.csv", dtype={
    "Path": str,
    "Atelectasis": np.float32,
    "Cardiomegaly": np.float32,
    "Consolidation": np.float32,
    "Edema": np.float32,
    "Pleural Effusion": np.float32,
    "Pleural Other": np.float32,
    "Pneumonia": np.float32,
    "Pneumothorax": np.float32,
    "Enlarged Cardiomediastinum": np.float32,
    "Lung Opacity": np.float32,
    "Lung Lesion": np.float32,
    "Fracture": np.float32,
    "Support Devices": np.float32,
    "No Finding": np.float32
})

In [5]:
list_columns = list(train_df.columns)
y_cols = list_columns[1::]  # First column is 'Path' column

In [6]:
train_datagen = ImageDataGenerator(
        featurewise_center=True,  # Mean and standard deviation values of the training set will be loaded to the object
        featurewise_std_normalization=True,
        rotation_range=10,
        shear_range=0.1,
        zoom_range=0.1,
        cval=0.0,
        fill_mode='constant',
        horizontal_flip=False,  # Some labels would be heavily affected by this change if it is True
        vertical_flip=False  # Not suitable for Chest X-ray images if it is True
    )

In [9]:
train_datagenerator = train_datagen.flow_from_dataframe(
        dataframe=train_df,
        directory=data_dir,
        x_col='Path',
        y_col=y_cols,
        target_size=(300, 300),
        color_mode='grayscale',
        class_mode='raw',
        batch_size=batch_size,
        shuffle=True,
        validate_filenames=True
    )

Found 178731 validated image filenames.


In [11]:
val_datagen = ImageDataGenerator(
        featurewise_center=True,  # Mean and standard deviation values of the training set will be loaded to the object
        featurewise_std_normalization=True
    )


In [12]:
val_datagenerator = val_datagen.flow_from_dataframe(
        dataframe=val_df,
        directory=data_dir,
        x_col='Path',
        y_col=y_cols,
        target_size=(300, 300),
        color_mode='grayscale',
        class_mode='raw',
        batch_size=batch_size,
        shuffle=False,
        validate_filenames=True
    )

Found 44683 validated image filenames.


In [17]:
gpu_devices = [""]*num_gpus

for i in range(num_gpus):
    gpu_devices[i] = f"/gpu:{i}"

In [18]:
gpu_devices

['/gpu:0', '/gpu:1', '/gpu:2', '/gpu:3']

In [48]:
optimizer = optimizers.Adam(
    lr=learning_rate,
    beta_1=0.9,
    beta_2=0.999,
    epsilon=0.1
)

/home/btech/nityanand.mathur/anaconda3/envs/tf/lib/python3.9/site-packages/keras/optimizers/optimizer_v2/adam.py:114: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  super().__init__(name, **kwargs)


In [22]:
positive_weights, negative_weights = compute_class_weights(labels=train_datagenerator.labels)
print(f"\nPositive Weights: {positive_weights}")
print(f"Negative Weights: {negative_weights}\n")


Positive Weights: [0.85008197 0.87875075 0.93383912 0.76508272 0.61434782 0.9843284
 0.97304329 0.91324393 0.95159765 0.52833028 0.95891591 0.95945303
 0.48110848 0.90044256]
Negative Weights: [0.14991803 0.12124925 0.06616088 0.23491728 0.38565218 0.0156716
 0.02695671 0.08675607 0.04840235 0.47166972 0.04108409 0.04054697
 0.51889152 0.09955744]



In [23]:
loss = set_binary_crossentropy_weighted_loss(
        positive_weights=positive_weights,
        negative_weights=negative_weights,
        epsilon=1e-7
    )

In [24]:
model_path = "MobileNetV2.h5"

In [49]:
def build_model(freeze=False):
    base_model = MobileNetV2(
        weights=None,
        include_top=False,
        input_shape=(300,300,1)
    )
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(2048, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)
    predictions = Dense(14, activation="sigmoid")(x)
    model = Model(inputs=base_model.input, outputs=predictions)

    if freeze:
        for layer in base_model.layers:
            layer.trainable=False
    
    return model   

In [50]:
strategy = tf.distribute.MirroredStrategy(devices=gpu_devices)
with strategy.scope():
    auc = tf.keras.metrics.AUC(
        name="auc",
        multi_label=True
    )
    model = build_model()
    model.compile(optimizer=optimizer, metrics=[auc,"accuracy"], loss=loss)


INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1', '/job:localhost/replica:0/task:0/device:GPU:2', '/job:localhost/replica:0/task:0/device:GPU:3')


In [51]:
print(model.summary())

Model: "model_3"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_7 (InputLayer)           [(None, 300, 300, 1  0           []                               
                                )]                                                                
                                                                                                  
 Conv1 (Conv2D)                 (None, 150, 150, 32  288         ['input_7[0][0]']                
                                )                                                                 
                                                                                                  
 bn_Conv1 (BatchNormalization)  (None, 150, 150, 32  128         ['Conv1[0][0]']                  
                                )                                                           

In [52]:
reducelronplateau = ReduceLROnPlateau(
        monitor="val_auc",
        factor=0.1,
        patience=5,
        verbose=1,
        mode="max",
        min_lr=1e-6
    )

checkpoint = ModelCheckpoint(
    filepath=model_path,
    monitor='val_auc',
    verbose=1,
    save_best_only=True,
    save_weights_only=False,
    mode='max'
)

In [53]:
fit = model.fit(
        x=train_datagenerator,
        epochs=training_epochs,
        validation_data=val_datagenerator,
        verbose=1,
        callbacks=[reducelronplateau, checkpoint]
    )

/home/btech/nityanand.mathur/anaconda3/envs/tf/lib/python3.9/site-packages/keras/preprocessing/image.py:1863: UserWarning: This ImageDataGenerator specifies `featurewise_center`, but it hasn't been fit on any training data. Fit it first by calling `.fit(numpy_data)`.
  warnings.warn(
/home/btech/nityanand.mathur/anaconda3/envs/tf/lib/python3.9/site-packages/keras/preprocessing/image.py:1873: UserWarning: This ImageDataGenerator specifies `featurewise_std_normalization`, but it hasn't been fit on any training data. Fit it first by calling `.fit(numpy_data)`.
  warnings.warn(
2022-11-08 00:27:45.095071: W tensorflow/core/grappler/optimizers/data/auto_shard.cc:776] AUTO sharding policy will apply DATA sharding policy as it failed to apply FILE sharding policy because of the following reason: Did not find a shardable source, walked to a node which is not a dataset: name: "FlatMapDataset/_2"
op: "FlatMapDataset"
input: "TensorDataset/_1"
attr {
  key: "Targuments"
  value {
    list {
    }

Epoch 1/30
INFO:tensorflow:batch_all_reduce: 162 all-reduces with algorithm = nccl, num_packs = 1
INFO:tensorflow:batch_all_reduce: 162 all-reduces with algorithm = nccl, num_packs = 1


2022-11-08 00:29:02.941865: I tensorflow/stream_executor/cuda/cuda_dnn.cc:384] Loaded cuDNN version 8100
2022-11-08 00:29:06.809916: I tensorflow/stream_executor/cuda/cuda_dnn.cc:384] Loaded cuDNN version 8100
2022-11-08 00:29:06.876067: W tensorflow/stream_executor/gpu/asm_compiler.cc:111] *** WARNING *** You are using ptxas 11.0.221, which is older than 11.1. ptxas before 11.1 is known to miscompile XLA code, leading to incorrect results or invalid-address errors.

You may not need to update to CUDA 11.1; cherry-picking the ptxas binary is often sufficient.
2022-11-08 00:29:10.344382: W tensorflow/stream_executor/gpu/asm_compiler.cc:111] *** WARNING *** You are using ptxas 11.0.221, which is older than 11.1. ptxas before 11.1 is known to miscompile XLA code, leading to incorrect results or invalid-address errors.

You may not need to update to CUDA 11.1; cherry-picking the ptxas binary is often sufficient.
2022-11-08 00:29:12.866168: I tensorflow/stream_executor/cuda/cuda_dnn.cc:384]

5586/5586 [==============================] - ETA: 0s - loss: 2.1330 - auc: 0.6233 - accuracy: 0.1256

2022-11-08 01:01:18.250024: W tensorflow/core/grappler/optimizers/data/auto_shard.cc:776] AUTO sharding policy will apply DATA sharding policy as it failed to apply FILE sharding policy because of the following reason: Did not find a shardable source, walked to a node which is not a dataset: name: "FlatMapDataset/_2"
op: "FlatMapDataset"
input: "TensorDataset/_1"
attr {
  key: "Targuments"
  value {
    list {
    }
  }
}
attr {
  key: "_cardinality"
  value {
    i: -2
  }
}
attr {
  key: "f"
  value {
    func {
      name: "__inference_Dataset_flat_map_flat_map_fn_243646"
    }
  }
}
attr {
  key: "metadata"
  value {
    s: "\n\021FlatMapDataset:81"
  }
}
attr {
  key: "output_shapes"
  value {
    list {
      shape {
        dim {
          size: -1
        }
        dim {
          size: -1
        }
        dim {
          size: -1
        }
        dim {
          size: -1
        }
      }
      shape {
        dim {
          size: -1
        }
        dim {
          size: 


Epoch 1: val_auc improved from -inf to 0.64034, saving model to MobileNetV2.h5
5586/5586 [==============================] - 2120s 361ms/step - loss: 2.1330 - auc: 0.6233 - accuracy: 0.1256 - val_loss: 2.7376 - val_auc: 0.6403 - val_accuracy: 0.0112 - lr: 0.0100
Epoch 2/30
5586/5586 [==============================] - ETA: 0s - loss: 1.9690 - auc: 0.6699 - accuracy: 0.1663
Epoch 2: val_auc improved from 0.64034 to 0.70101, saving model to MobileNetV2.h5
5586/5586 [==============================] - 2340s 419ms/step - loss: 1.9690 - auc: 0.6699 - accuracy: 0.1663 - val_loss: 2.6972 - val_auc: 0.7010 - val_accuracy: 0.1976 - lr: 0.0100
Epoch 3/30
5586/5586 [==============================] - ETA: 0s - loss: 1.8848 - auc: 0.6965 - accuracy: 0.1890
Epoch 3: val_auc improved from 0.70101 to 0.72435, saving model to MobileNetV2.h5
5586/5586 [==============================] - 2336s 418ms/step - loss: 1.8848 - auc: 0.6965 - accuracy: 0.1890 - val_loss: 1.9845 - val_auc: 0.7243 - val_accuracy: 0.1

In [61]:
# wandb.config.update(config)

In [63]:
wandb.log(fit.history)